<a href="https://colab.research.google.com/github/12halima/Transport_Recommander/blob/main/notebooks/analyze_stop_times_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


In [ ]:
import os

# ✅ Augmenter la mémoire pour Spark (important pour 31M lignes)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 8g --executor-memory 8g pyspark-shell"

In [ ]:
import os

# ✅ Augmenter la mémoire Spark
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 8g --executor-memory 8g pyspark-shell"

from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, regexp_extract

spark = SparkSession.builder \
    .appName("GTFS Stop Times Analysis") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder.getOrCreate()

base_path = "/content/drive/MyDrive/GTFS_CLEAN"   # 👉 Ton dossier contenant les sous-dossiers GTFS

dfs = []

# Recherche de tous les fichiers stop_times.txt dans les sous-dossiers
for root, dirs, files in os.walk(base_path):
    if "stop_times.txt" in files:
        file_path = os.path.join(root, "stop_times.txt")
        print("📄 Fichier trouvé :", file_path)

        df = spark.read.option("header", True).csv(file_path)
        dfs.append(df)

# Vérification si aucun fichier trouvé
if not dfs:
    raise Exception("Aucun fichier stop_times.txt trouvé dans /content/drive/MyDrive/GTFS_CLEAN!")

# Afficher les colonnes de chaque DataFrame pour vérifier les différences
print("\n🔎 Vérification des colonnes :\n")
for i, df in enumerate(dfs):
    print(f"Fichier {i+1} colonnes : {df.columns}")

# Fusion des DataFrames même si les colonnes ne sont pas identiques
df_stop_times = dfs[0]
for df in dfs[1:]:
    df_stop_times = df_stop_times.unionByName(df, allowMissingColumns=True)

print("\n✅ Fusion terminée !")
print("Nombre total de lignes :", df_stop_times.count())
df_stop_times.printSchema()

# Aperçu final
df_stop_times.show(10)


📄 Fichier trouvé : /content/drive/MyDrive/GTFS_CLEAN/GTFS_CLEAN/Vectalia Movilidad (bus de la ville de Cáceres)/stop_times.txt
📄 Fichier trouvé : /content/drive/MyDrive/GTFS_CLEAN/GTFS_CLEAN/Xunta de Galicia Buses/stop_times.txt
📄 Fichier trouvé : /content/drive/MyDrive/GTFS_CLEAN/GTFS_CLEAN/Àrea Metropolitana de Barcelona (AMB)/stop_times.txt
📄 Fichier trouvé : /content/drive/MyDrive/GTFS_CLEAN/GTFS_CLEAN/Viagón Coaches/stop_times.txt
📄 Fichier trouvé : /content/drive/MyDrive/GTFS_CLEAN/GTFS_CLEAN/TUSSAM (Seville bus and tram)/stop_times.txt
📄 Fichier trouvé : /content/drive/MyDrive/GTFS_CLEAN/GTFS_CLEAN/TUS (Transportes Urbanos de Santander)/stop_times.txt
📄 Fichier trouvé : /content/drive/MyDrive/GTFS_CLEAN/GTFS_CLEAN/Transports Municipals del Gironés SAU (TMG) Girona city bus/stop_times.txt
📄 Fichier trouvé : /content/drive/MyDrive/GTFS_CLEAN/GTFS_CLEAN/Transports Municipaux D’Egara (TMESA) Terrassa bus urbain/stop_times.txt
📄 Fichier trouvé : /content/drive/MyDrive/GTFS_CLEAN/

In [ ]:
# 4️⃣ Ajouter la colonne 'source_folder' pour savoir de quel sous-dossier vient chaque ligne
df_stop_times = df_stop_times.withColumn(
    "source_folder",
    regexp_extract(input_file_name(), r"/([^/]+)/stop_times.txt", 1)
)

In [ ]:
# 5️⃣ Afficher un aperçu
df_stop_times.show(5, truncate=False)

+-----------------+------------+--------------+-------+-------------+-----------+-------------+-------------------+----------+----------+----------+----------+----------+-----------+-----------+-----------+-----------+-----------+-----------+-------------+-------------------+---------+-----------------+----+-----------+--------------------------------------------------------------+
|trip_id          |arrival_time|departure_time|stop_id|stop_sequence|pickup_type|drop_off_type|continuous_drop_off|Unnamed: 5|Unnamed: 6|Unnamed: 7|Unnamed: 8|Unnamed: 9|Unnamed: 10|Unnamed: 11|Unnamed: 12|Unnamed: 13|Unnamed: 14|Unnamed: 15|stop_headsign|shape_dist_traveled|timepoint|continuous_pickup|_c8 |prohibition|source_folder                                                 |
+-----------------+------------+--------------+-------+-------------+-----------+-------------+-------------------+----------+----------+----------+----------+----------+-----------+-----------+-----------+-----------+-----------+

In [ ]:
# 6️⃣ Compter le nombre total de lignes pour vérifier
print("Nombre total de lignes :", df_stop_times.count())

Nombre total de lignes : 33931494


In [ ]:
# Afficher 20 lignes de la colonne source_folder
df_stop_times.select("source_folder").show(20, truncate=False)


+--------------------------------------------------------------+
|source_folder                                                 |
+--------------------------------------------------------------+
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(bus%20de%20la%20ville%20de%20Cáceres)|
|Vectalia%20Movilidad%20(

In [ ]:
import urllib.parse


In [ ]:
import urllib.parse
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Décoder les noms encodés (%20 -> espace)
decode_udf = udf(lambda x: urllib.parse.unquote(x) if x else x, StringType())
df_stop_times = df_stop_times.withColumn("source_folder", decode_udf("source_folder"))

# Vérifier les noms de dossiers différents
df_stop_times.select("source_folder").distinct().show(truncate=False)

# Afficher quelques lignes pour contrôle
df_stop_times.show(10, truncate=False)


+------------------------------------------------------------------------------------------------------------------------+
|source_folder                                                                                                           |
+------------------------------------------------------------------------------------------------------------------------+
|Vectalia Movilidad (bus de la ville de Cáceres)                                                                        |
|Xunta de Galicia Buses                                                                                                  |
|Àrea Metropolitana de Barcelona (AMB)                                                                                  |
|Viagón Coaches                                                                                                         |
|TUSSAM (Seville bus and tram)                                                                                           |
|TUS (Transporte

In [ ]:
df_stop_times.groupBy("source_folder").count().show(truncate=False)


+------------------------------------------------------------------------------------------------------------------------+-------+
|source_folder                                                                                                           |count  |
+------------------------------------------------------------------------------------------------------------------------+-------+
|Vectalia Movilidad (bus de la ville de Cáceres)                                                                        |290088 |
|Xunta de Galicia Buses                                                                                                  |3384398|
|Àrea Metropolitana de Barcelona (AMB)                                                                                  |980023 |
|Viagón Coaches                                                                                                         |240    |
|TUSSAM (Seville bus and tram)                                                     

In [ ]:
df_stop_times.columns

['trip_id',
 'arrival_time',
 'departure_time',
 'stop_id',
 'stop_sequence',
 'pickup_type',
 'drop_off_type',
 'continuous_drop_off',
 'Unnamed: 5',
 'Unnamed: 6',
 'Unnamed: 7',
 'Unnamed: 8',
 'Unnamed: 9',
 'Unnamed: 10',
 'Unnamed: 11',
 'Unnamed: 12',
 'Unnamed: 13',
 'Unnamed: 14',
 'Unnamed: 15',
 'stop_headsign',
 'shape_dist_traveled',
 'timepoint',
 'continuous_pickup',
 '_c8',
 'prohibition',
 'source_folder']

In [ ]:
df_stop_times = df_stop_times.drop("_c8", "timepoint","Unnamed: 5","Unnamed: 6","Unnamed: 7","Unnamed: 8","Unnamed: 9","Unnamed: 10","Unnamed: 11","Unnamed: 12","Unnamed: 13","Unnamed: 14","Unnamed: 15")


In [ ]:
from pyspark.sql.functions import col, trim, when, count

colonnes_obligatoires = ["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"]

resultats = {}

for c in colonnes_obligatoires:
    resultats[c] = {
        "null": df_stop_times.filter(col(c).isNull()).count(),
        "vide_ou_espace": df_stop_times.filter(trim(col(c)) == "").count()
    }

# Affichage
import pprint
pprint.pprint(resultats)


{'arrival_time': {'null': 2170463, 'vide_ou_espace': 9160},
 'departure_time': {'null': 2170463, 'vide_ou_espace': 9160},
 'stop_id': {'null': 0, 'vide_ou_espace': 0},
 'stop_sequence': {'null': 0, 'vide_ou_espace': 0},
 'trip_id': {'null': 0, 'vide_ou_espace': 0}}


In [ ]:
from pyspark.sql.functions import col, trim

# Liste des colonnes obligatoires
mandatory_cols = ["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"]

# Nettoyer : supprimer les lignes avec null, NaN ou vide/espaces
df_clean = df_stop_times

for c in mandatory_cols:
    df_clean = df_clean.filter((col(c).isNotNull()) & (trim(col(c)) != ""))

# Vérification : afficher nombre de lignes avant et après nettoyage
print("Nombre de lignes avant nettoyage :", df_stop_times.count())
print("Nombre de lignes après nettoyage :", df_clean.count())

# Afficher un échantillon
df_clean.select(mandatory_cols + ["source_folder"]).show(20, truncate=False)


Nombre de lignes avant nettoyage : 33931494
Nombre de lignes après nettoyage : 31751871
+-----------------+------------+--------------+-------+-------------+------------------------------------------------+
|trip_id          |arrival_time|departure_time|stop_id|stop_sequence|source_folder                                   |
+-----------------+------------+--------------+-------+-------------+------------------------------------------------+
|2855_1_704_339306|07:05:00    |07:05:00      |2      |0            |Vectalia Movilidad (bus de la ville de Cáceres)|
|2855_1_704_339306|07:08:00    |07:08:00      |3      |1            |Vectalia Movilidad (bus de la ville de Cáceres)|
|2855_1_704_339306|07:10:00    |07:10:00      |4      |2            |Vectalia Movilidad (bus de la ville de Cáceres)|
|2855_1_704_339306|07:12:00    |07:12:00      |5      |3            |Vectalia Movilidad (bus de la ville de Cáceres)|
|2855_1_704_339306|07:14:00    |07:14:00      |6      |4            |Vectalia M

In [ ]:
from pyspark.sql.functions import col, trim, isnan, sum as spark_sum

# Liste des colonnes obligatoires
colonnes_obligatoires = ["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"]

# Vérifier null, NaN et vide pour chaque colonne obligatoire
controle = df_clean.select([
    spark_sum(col(c).isNull().cast("int")).alias(f"{c}_null") for c in colonnes_obligatoires
] + [
    spark_sum(isnan(col(c)).cast("int")).alias(f"{c}_nan") for c in colonnes_obligatoires
] + [
    spark_sum((trim(col(c)) == "").cast("int")).alias(f"{c}_vide") for c in colonnes_obligatoires
])

controle.show(truncate=False)


+------------+-----------------+-------------------+------------+------------------+-----------+----------------+------------------+-----------+-----------------+------------+-----------------+-------------------+------------+------------------+
|trip_id_null|arrival_time_null|departure_time_null|stop_id_null|stop_sequence_null|trip_id_nan|arrival_time_nan|departure_time_nan|stop_id_nan|stop_sequence_nan|trip_id_vide|arrival_time_vide|departure_time_vide|stop_id_vide|stop_sequence_vide|
+------------+-----------------+-------------------+------------+------------------+-----------+----------------+------------------+-----------+-----------------+------------+-----------------+-------------------+------------+------------------+
|0           |0                |0                  |0           |0                 |0          |0               |0                 |0          |0                |0           |0                |0                  |0           |0                 |
+------------+--

In [ ]:
colonnes_supplementaires = ["pickup_type", "drop_off_type","shape_dist_traveled","stop_headsign","continuous_pickup","continuous_drop_off","timepoint","prohibition"]


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import isnan

total_lignes = df_clean.count()

resultats = []

for col in colonnes_supplementaires:
    nb_null = df_clean.filter(F.col(col).isNull()).count()
    nb_nan = df_clean.filter(isnan(col)).count()
    nb_vide = df_clean.filter(F.col(col) == "").count()

    # Forcer l'usage du round Python
    null_pct = float(__builtins__.round(nb_null / total_lignes * 100, 2))
    nan_pct  = float(__builtins__.round(nb_nan  / total_lignes * 100, 2))
    vide_pct = float(__builtins__.round(nb_vide / total_lignes * 100, 2))

    resultats.append((
        col,
        nb_null, null_pct,
        nb_nan, nan_pct,
        nb_vide, vide_pct
    ))

schema = [
    "colonne",
    "null", "null_%",
    "nan", "nan_%",
    "vide", "vide_%"
]

df_supp = spark.createDataFrame(resultats, schema)
df_supp.show(truncate=False)


+-------------------+--------+------+---+-----+----+------+
|colonne            |null    |null_%|nan|nan_%|vide|vide_%|
+-------------------+--------+------+---+-----+----+------+
|pickup_type        |7197738 |22.67 |0  |0.0  |0   |0.0   |
|drop_off_type      |7352711 |23.16 |0  |0.0  |0   |0.0   |
|shape_dist_traveled|20200529|63.62 |0  |0.0  |0   |0.0   |
|stop_headsign      |30327561|95.51 |0  |0.0  |0   |0.0   |
|continuous_pickup  |31014162|97.68 |0  |0.0  |0   |0.0   |
|continuous_drop_off|30034139|94.59 |0  |0.0  |0   |0.0   |
|timepoint          |27731818|87.34 |0  |0.0  |0   |0.0   |
|prohibition        |31335200|98.69 |0  |0.0  |0   |0.0   |
+-------------------+--------+------+---+-----+----+------+



In [ ]:
df_clean = df_clean.drop("shape_dist_traveled", "timepoint","prohibition","stop_headsign","continuous_pickup","continuous_drop_off")


In [ ]:
df_clean.columns


['trip_id',
 'arrival_time',
 'departure_time',
 'stop_id',
 'stop_sequence',
 'pickup_type',
 'drop_off_type',
 'source_folder']

In [ ]:
from pyspark.sql.functions import col, when, lit, trim, isnan

df_clean = df_clean.withColumn(
    "pickup_type",
    when(
        (col("pickup_type").isNull()) |      # null
        (col("pickup_type") == "") |        # chaîne vide
        (trim(col("pickup_type")) == "") |  # espaces
        (col("pickup_type") == "nan") |     # texte "nan"
        (isnan(col("pickup_type"))),         # NaN numérique
        lit(0)
    ).otherwise(col("pickup_type"))
)


In [ ]:
from pyspark.sql.types import IntegerType
df_clean = df_clean.withColumn("pickup_type", col("pickup_type").cast(IntegerType()))


In [ ]:
df_clean.select("pickup_type").distinct().show(truncate=False)


+-----------+
|pickup_type|
+-----------+
|0          |
|2          |
|1          |
|3          |
+-----------+



In [ ]:
from pyspark.sql.functions import col, when, lit

df_clean = df_clean.withColumn(
    "pickup_type",
    when(col("pickup_type").isin([0, 1, 2, 3]), col("pickup_type")) \
    .otherwise(lit(0))
)


In [ ]:
df_clean.select("pickup_type").distinct().show()


+-----------+
|pickup_type|
+-----------+
|          0|
|          2|
|          1|
|          3|
+-----------+



In [ ]:
from pyspark.sql.functions import col, when, lit, trim, isnan

df_clean = df_clean.withColumn(
    "drop_off_type",
    when(
        (col("drop_off_type").isNull()) |      # null
        (col("drop_off_type") == "") |        # chaîne vide
        (trim(col("drop_off_type")) == "") |  # espaces
        (col("drop_off_type") == "nan") |     # texte "nan"
        (isnan(col("drop_off_type"))),         # NaN numérique
        lit(0)
    ).otherwise(col("drop_off_type"))
)

In [ ]:
# Vérifier les valeurs uniques pour confirmer
df_clean.select("drop_off_type").distinct().show(20)

+-------------+
|drop_off_type|
+-------------+
|            0|
|            3|
|            1|
|            2|
+-------------+



In [ ]:
from pyspark.sql.functions import col, when

# Limiter drop_off_type aux codes 0,1,2,3
df_clean = df_clean.withColumn(
    "drop_off_type",
    when(col("drop_off_type").isin([0, 1, 2, 3]), col("drop_off_type"))
    .otherwise(0)
)

In [ ]:
from pyspark.sql.functions import col, when

# Limiter drop_off_type aux codes 0,1,2,3
df_clean = df_clean.withColumn(
    "drop_off_type",
    when(col("drop_off_type").isin([0, 1, 2, 3]), col("drop_off_type"))
    .otherwise(0)
)

In [ ]:
# Vérifier les valeurs uniques pour confirmer
df_clean.select("drop_off_type").distinct().show(20)

+-------------+
|drop_off_type|
+-------------+
|            0|
|            3|
|            1|
|            2|
+-------------+



In [ ]:
from pyspark.sql.functions import col

# Transformer les colonnes en entier
df_clean = df_clean.withColumn("pickup_type", col("pickup_type").cast("int"))
df_clean = df_clean.withColumn("drop_off_type", col("drop_off_type").cast("int"))

# Vérifier le type
df_clean.printSchema()


root
 |-- trip_id: string (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- stop_sequence: string (nullable = true)
 |-- pickup_type: integer (nullable = true)
 |-- drop_off_type: integer (nullable = true)
 |-- source_folder: string (nullable = true)



In [ ]:
df_clean.select("trip_id").sample(fraction=0.001).show(20, truncate=False)


+-----------------+
|trip_id          |
+-----------------+
|2855_1_704_339446|
|2855_1_704_339474|
|2855_1_704_339480|
|2855_1_704_339533|
|2855_1_704_339541|
|2855_1_704_339551|
|2855_1_704_339556|
|2855_1_704_339606|
|2855_1_704_339732|
|2855_1_704_339786|
|2855_1_704_339905|
|2855_1_704_339927|
|2855_1_704_339956|
|2855_1_704_339997|
|2855_1_704_340000|
|2855_1_704_340004|
|2855_1_704_340036|
|2855_1_704_340146|
|2855_1_704_340331|
|2855_1_705_339421|
+-----------------+
only showing top 20 rows



In [ ]:
from pyspark.sql.functions import col, lower, trim, regexp_replace

df_clean = df_clean.withColumn(
    "stop_id",
    trim(lower(regexp_replace(col("stop_id"), r"\s+", " ")))  # minuscule + enlever les espaces multiples
)


In [ ]:
from pyspark.sql.functions import col, lower, trim, regexp_replace

df_clean = df_clean.withColumn(
    "stop_id",
    trim(lower(regexp_replace(col("stop_id"), r"\s+", " ")))  # minuscule + enlever les espaces multiples
)


In [ ]:
from pyspark.sql.functions import col, lower, trim, regexp_replace

df_clean = df_clean.withColumn(
    "trip_id",
    trim(lower(regexp_replace(col("trip_id"), r"\s+", " ")))  # minuscule + enlever les espaces multiples
)


In [ ]:
df_clean.select("stop_id").show(20, truncate=False)

+-------+
|stop_id|
+-------+
|2      |
|3      |
|4      |
|5      |
|6      |
|7      |
|8      |
|240    |
|241    |
|14     |
|160    |
|15     |
|15     |
|16     |
|297    |
|17     |
|18     |
|19     |
|20     |
|21     |
+-------+
only showing top 20 rows



In [ ]:
df_clean.select("trip_id").show(20, truncate=False)


+-----------------+
|trip_id          |
+-----------------+
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
+-----------------+
only showing top 20 rows



In [ ]:
from pyspark.sql.functions import regexp_replace

df_clean = df_clean.withColumn(
    "trip_id",
    regexp_replace(col("trip_id"), r"[^a-z0-9_\-]", "")  # garder seulement lettres, chiffres, _ et -
)


In [ ]:
df_clean.select("trip_id").show(20, truncate=False)

+-----------------+
|trip_id          |
+-----------------+
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339306|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
|2855_1_704_339307|
+-----------------+
only showing top 20 rows



In [ ]:
df_clean.select("arrival_time").sample(fraction=0.001).show(20, truncate=False)


+------------+
|arrival_time|
+------------+
|22:09:00    |
|21:55:00    |
|13:38:00    |
|15:14:00    |
|07:50:00    |
|21:39:00    |
|21:18:00    |
|15:51:00    |
|17:55:00    |
|21:43:00    |
|16:25:00    |
|22:00:00    |
|22:23:00    |
|18:54:00    |
|09:45:00    |
|14:04:00    |
|12:56:00    |
|10:45:00    |
|22:10:00    |
|08:57:00    |
+------------+
only showing top 20 rows



In [ ]:
from pyspark.sql.functions import col, trim

# enlever espaces autour
df_clean = df_clean.withColumn("arrival_time", trim(col("arrival_time")))

# filtrer les formats invalides (tout ce qui ne correspond pas à HH:MM:SS)
invalid_times = df_clean.filter(~col("arrival_time").rlike(r"^[0-2][0-9]:[0-5][0-9]:[0-5][0-9]$"))

# afficher un échantillon
invalid_times.select("trip_id", "arrival_time").show(20, truncate=False)

# compter le nombre total de valeurs invalides
nb_invalid = invalid_times.count()
print(f"Nombre de formats invalides dans arrival_time : {nb_invalid}")


+-------+------------+
|trip_id|arrival_time|
+-------+------------+
|1288110|9:00:00     |
|1288110|9:02:00     |
|1288110|9:04:00     |
|1288110|9:05:36     |
|1288110|9:07:00     |
|1288110|9:08:00     |
|1288110|9:09:00     |
|1288110|9:10:00     |
|1288110|9:12:00     |
|1288110|9:13:00     |
|1288110|9:20:00     |
|1288110|9:21:00     |
|1288110|9:22:00     |
|1288110|9:23:00     |
|1288110|9:24:00     |
|1288110|9:25:00     |
|1288110|9:26:00     |
|1288110|9:27:00     |
|1288110|9:28:00     |
|1288110|9:29:42     |
+-------+------------+
only showing top 20 rows

Nombre de formats invalides dans arrival_time : 227091


In [ ]:
from pyspark.sql.functions import split, concat_ws, col

# Séparer HH, MM, SS
time_parts = split(col("arrival_time"), ":")

# Nouveau HH = HH % 24, garder MM et SS
df_clean = df_clean.withColumn(
    "arrival_time",
    concat_ws(":",
              (time_parts.getItem(0).cast("int") % 24).cast("string"),
              time_parts.getItem(1),
              time_parts.getItem(2)
             )
)


In [ ]:
from pyspark.sql.functions import col, trim

# enlever espaces autour
df_clean = df_clean.withColumn("arrival_time", trim(col("arrival_time")))

# filtrer les formats invalides (tout ce qui ne correspond pas à HH:MM:SS)
invalid_times = df_clean.filter(~col("arrival_time").rlike(r"^[0-2][0-9]:[0-5][0-9]:[0-5][0-9]$"))

# afficher un échantillon
invalid_times.select("trip_id", "arrival_time").show(20, truncate=False)

# compter le nombre total de valeurs invalides
nb_invalid = invalid_times.count()
print(f"Nombre de formats invalides dans arrival_time : {nb_invalid}")


+-----------------+------------+
|trip_id          |arrival_time|
+-----------------+------------+
|2855_1_704_339306|7:05:00     |
|2855_1_704_339306|7:08:00     |
|2855_1_704_339306|7:10:00     |
|2855_1_704_339306|7:12:00     |
|2855_1_704_339306|7:14:00     |
|2855_1_704_339306|7:15:00     |
|2855_1_704_339306|7:21:00     |
|2855_1_704_339306|7:22:00     |
|2855_1_704_339306|7:24:00     |
|2855_1_704_339306|7:24:00     |
|2855_1_704_339306|7:26:00     |
|2855_1_704_339306|7:30:00     |
|2855_1_704_339307|7:30:00     |
|2855_1_704_339307|7:30:00     |
|2855_1_704_339307|7:31:00     |
|2855_1_704_339307|7:32:00     |
|2855_1_704_339307|7:39:00     |
|2855_1_704_339307|7:40:00     |
|2855_1_704_339307|7:43:00     |
|2855_1_704_339307|7:45:00     |
+-----------------+------------+
only showing top 20 rows

Nombre de formats invalides dans arrival_time : 8009623


In [ ]:
from pyspark.sql.functions import col, split, concat_ws

# Séparer HH, MM, SS
time_parts = split(col("arrival_time"), ":")

# Appliquer modulo 24 sur HH
df_clean = df_clean.withColumn(
    "arrival_time",
    concat_ws(":",
              (time_parts.getItem(0).cast("int") % 24).cast("string"),
              time_parts.getItem(1),
              time_parts.getItem(2)
             )
)


In [ ]:
nb_invalid = df_clean.filter(
    ~col("arrival_time").rlike(r"^([0-1]?[0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$")
).count()

print("Nombre de formats invalides après correction :", nb_invalid)


Nombre de formats invalides après correction : 0


In [ ]:
from pyspark.sql.functions import col, trim

# enlever espaces autour
df_clean = df_clean.withColumn("departure_time", trim(col("departure_time")))

# filtrer les formats invalides (tout ce qui ne correspond pas à HH:MM:SS)
invalid_times = df_clean.filter(~col("departure_time").rlike(r"^[0-2][0-9]:[0-5][0-9]:[0-5][0-9]$"))

# afficher un échantillon
invalid_times.select("trip_id", "departure_time").show(20, truncate=False)

# compter le nombre total de valeurs invalides
nb_invalid = invalid_times.count()
print(f"Nombre de formats invalides dans arrival_time : {nb_invalid}")


+-------+--------------+
|trip_id|departure_time|
+-------+--------------+
|1288110|9:00:00       |
|1288110|9:02:00       |
|1288110|9:04:00       |
|1288110|9:05:36       |
|1288110|9:07:00       |
|1288110|9:08:00       |
|1288110|9:09:00       |
|1288110|9:10:00       |
|1288110|9:12:00       |
|1288110|9:13:00       |
|1288110|9:20:00       |
|1288110|9:21:00       |
|1288110|9:22:00       |
|1288110|9:23:00       |
|1288110|9:24:00       |
|1288110|9:25:00       |
|1288110|9:26:00       |
|1288110|9:27:00       |
|1288110|9:28:00       |
|1288110|9:29:42       |
+-------+--------------+
only showing top 20 rows

Nombre de formats invalides dans arrival_time : 227167


In [ ]:
from pyspark.sql.functions import col, split, concat_ws

# Séparer HH, MM, SS
time_parts = split(col("departure_time"), ":")

# Appliquer modulo 24 sur HH
df_clean = df_clean.withColumn(
    "arrival_time",
    concat_ws(":",
              (time_parts.getItem(0).cast("int") % 24).cast("string"),
              time_parts.getItem(1),
              time_parts.getItem(2)
             )
)


In [ ]:
nb_invalid = df_clean.filter(
    ~col("departure_time").rlike(r"^([0-1]?[0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$")
).count()

print("Nombre de formats invalides après correction :", nb_invalid)


Nombre de formats invalides après correction : 739799


In [ ]:
from pyspark.sql.functions import col, trim

# enlever espaces autour
df_clean = df_clean.withColumn("departure_time", trim(col("departure_time")))

# filtrer les formats invalides (tout ce qui ne correspond pas à HH:MM:SS)
invalid_times = df_clean.filter(~col("departure_time").rlike(r"^[0-2][0-9]:[0-5][0-9]:[0-5][0-9]$"))

# afficher un échantillon
invalid_times.select("trip_id", "departure_time").show(20, truncate=False)

# compter le nombre total de valeurs invalides
nb_invalid = invalid_times.count()
print(f"Nombre de formats invalides dans arrival_time : {nb_invalid}")


+-------+--------------+
|trip_id|departure_time|
+-------+--------------+
|1288110|9:00:00       |
|1288110|9:02:00       |
|1288110|9:04:00       |
|1288110|9:05:36       |
|1288110|9:07:00       |
|1288110|9:08:00       |
|1288110|9:09:00       |
|1288110|9:10:00       |
|1288110|9:12:00       |
|1288110|9:13:00       |
|1288110|9:20:00       |
|1288110|9:21:00       |
|1288110|9:22:00       |
|1288110|9:23:00       |
|1288110|9:24:00       |
|1288110|9:25:00       |
|1288110|9:26:00       |
|1288110|9:27:00       |
|1288110|9:28:00       |
|1288110|9:29:42       |
+-------+--------------+
only showing top 20 rows

Nombre de formats invalides dans arrival_time : 227167


In [ ]:
from pyspark.sql.functions import col, split, concat_ws

# Séparer HH, MM, SS
time_parts = split(col("departure_time"), ":")

# Appliquer modulo 24 sur HH
df_clean = df_clean.withColumn(
    "arrival_time",
    concat_ws(":",
              (time_parts.getItem(0).cast("int") % 24).cast("string"),
              time_parts.getItem(1),
              time_parts.getItem(2)
             )
)


In [ ]:
nb_invalid = df_clean.filter(
    ~col("departure_time").rlike(r"^([0-1]?[0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$")
).count()

print("Nombre de formats invalides après correction :", nb_invalid)

Nombre de formats invalides après correction : 739799


In [ ]:
from pyspark.sql import functions as F

# Séparer HH MM SS
df_clean = df_clean.withColumn(
    "parts",
    F.split(F.col("departure_time"), ":")
)

df_clean = df_clean.withColumn("hour",   F.col("parts")[0].cast("int"))
df_clean = df_clean.withColumn("minute", F.col("parts")[1].cast("int"))
df_clean = df_clean.withColumn("second", F.col("parts")[2].cast("int"))

# Convertir heures > 24 → modulo 24
df_clean = df_clean.withColumn(
    "hour_fixed",
    (F.col("hour") % 24)
)

# Reconstruire la colonne departure_time
df_clean = df_clean.withColumn(
    "departure_time",
    F.concat(
        F.lpad(F.col("hour_fixed"), 2, "0"),
        F.lit(":"),
        F.lpad(F.col("minute"), 2, "0"),
        F.lit(":"),
        F.lpad(F.col("second"), 2, "0")
    )
)

# Nettoyer les colonnes temporaires
df_clean = df_clean.drop("parts", "hour", "minute", "second", "hour_fixed")


In [ ]:
df_clean.select("departure_time").show(20, truncate=False)


+--------------+
|departure_time|
+--------------+
|07:05:00      |
|07:08:00      |
|07:10:00      |
|07:12:00      |
|07:14:00      |
|07:15:00      |
|07:21:00      |
|07:22:00      |
|07:24:00      |
|07:24:00      |
|07:26:00      |
|07:30:00      |
|07:30:00      |
|07:30:00      |
|07:31:00      |
|07:32:00      |
|07:39:00      |
|07:40:00      |
|07:43:00      |
|07:45:00      |
+--------------+
only showing top 20 rows



In [ ]:
nb_invalid = df_clean.filter(
    ~col("departure_time").rlike(r"^([0-1]?[0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$")
).count()

print("Nombre de formats invalides après correction :", nb_invalid)

Nombre de formats invalides après correction : 0


In [ ]:
# 5️⃣ Afficher un aperçu
df_clean.show(5, truncate=False)

+-----------------+------------+--------------+-------+-------------+-----------+-------------+------------------------------------------------+
|trip_id          |arrival_time|departure_time|stop_id|stop_sequence|pickup_type|drop_off_type|source_folder                                   |
+-----------------+------------+--------------+-------+-------------+-----------+-------------+------------------------------------------------+
|2855_1_704_339306|7:05:00     |07:05:00      |2      |0            |0          |0            |Vectalia Movilidad (bus de la ville de Cáceres)|
|2855_1_704_339306|7:08:00     |07:08:00      |3      |1            |0          |0            |Vectalia Movilidad (bus de la ville de Cáceres)|
|2855_1_704_339306|7:10:00     |07:10:00      |4      |2            |0          |0            |Vectalia Movilidad (bus de la ville de Cáceres)|
|2855_1_704_339306|7:12:00     |07:12:00      |5      |3            |0          |0            |Vectalia Movilidad (bus de la ville

In [ ]:
# 6️⃣ Compter le nombre total de lignes pour vérifier
print("Nombre total de lignes :", df_clean.count())

Nombre total de lignes : 31751871


In [ ]:
temp_path = "/content/drive/MyDrive/stop_times_tmp"

df_clean.write.mode("overwrite").option("header", True).csv(temp_path)

print("✔️ Écriture Spark terminée en partitions.")


✔️ Écriture Spark terminée en partitions.


In [2]:
import pandas as pd
import glob

path = "/content/drive/MyDrive/stop_times_tmp"

files = glob.glob(path + "/part-*.csv")

dfs = [pd.read_csv(f) for f in files]

df_final = pd.concat(dfs, ignore_index=True)

df_final.to_csv("/content/drive/MyDrive/GTFS_FINAL/stop_times_final.csv", index=False)
